---
## 0.1 Importaciones y configuración global

In [ ]:
import os, glob, warnings, re
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import stats
from scipy.signal import find_peaks, savgol_filter
from scipy.spatial.distance import cdist
from scipy.cluster.hierarchy import dendrogram, linkage

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (silhouette_score, silhouette_samples,
                              calinski_harabasz_score, davies_bouldin_score)
from sklearn.impute import SimpleImputer

try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("UMAP no disponible — pip install umap-learn")

plt.rcParams.update({
    'figure.dpi': 130, 'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f8', 'axes.grid': True,
    'grid.alpha': 0.35, 'grid.linestyle': '--',
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 10,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False,
})
SEED = 42
np.random.seed(SEED)

ESTADO_COLORS  = {'pregunta': '#4C72B0', 'respuesta': '#DD8452'}
CLUSTER_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

AU_NAMES = {
    'AU01_r': 'Elev. ceja medial',  'AU02_r': 'Elev. ceja lateral',
    'AU04_r': 'Fruncimiento ceño',  'AU05_r': 'Elev. párpado sup.',
    'AU06_r': 'Mejilla elevada',    'AU07_r': 'Tensión párpado',
    'AU09_r': 'Arrugamiento nariz', 'AU10_r': 'Elev. labio sup.',
    'AU12_r': 'Tirón comisura',     'AU14_r': 'Hoyuelo (dimpler)',
    'AU15_r': 'Depresor comisura',  'AU17_r': 'Elev. mentón',
    'AU20_r': 'Estiramiento labial','AU23_r': 'Tensión labios',
    'AU25_r': 'Apertura labios',    'AU26_r': 'Descenso mandíbula',
    'AU45_r': 'Parpadeo',
}
AU_COLS_R   = list(AU_NAMES.keys())
AU_COLS_C_ALL = [f'AU{n}_c' for n in
                 ['01','02','04','05','06','07','09','10',
                  '12','14','15','17','20','23','25','26','28','45']]
POSE_HEAD   = ['pose_Rx','pose_Ry','pose_Rz','pose_Tx','pose_Ty','pose_Tz']
GAZE_COLS   = ['gaze_angle_x','gaze_angle_y']
BODY_COLS_C = ['HombroIzq_X_c','HombroIzq_Y_c','HombroDer_X_c','HombroDer_Y_c',
               'CodoIzq_X_c','CodoIzq_Y_c','CodoDer_X_c','CodoDer_Y_c',
               'MunecaIzq_X_c','MunecaIzq_Y_c','MunecaDer_X_c','MunecaDer_Y_c']

def savefig(name):
    plt.savefig(os.path.join(OUTPUT_DIR, name), bbox_inches='tight', dpi=150)

print("Librerías cargadas. UMAP:", UMAP_AVAILABLE)


---
## 0.2 Carga del dataset completo

In [ ]:
DATA_DIR   = os.getcwd()     
OUTPUT_DIR = './figuras_cap5_6/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
print(f"CSVs encontrados: {len(csv_files)}")

dfs = []
for f in csv_files:
    tmp = pd.read_csv(f)
    tmp['clip_file'] = os.path.basename(f)
    num = int(re.search(r'(\d+)', os.path.basename(f)).group(1))
    tmp['sesion'] = 1 if num <= 29 else 2
    dfs.append(tmp)

df_all = pd.concat(dfs, ignore_index=True)
df_all.columns = df_all.columns.str.strip()
df_all['estado']     = df_all['estado'].astype(str).str.strip()
df_all['success']    = df_all['success'].astype(int)
df_all['confidence'] = pd.to_numeric(df_all['confidence'], errors='coerce')

print(f"\nDataset: {df_all.shape[0]:,} fotogramas x {df_all.shape[1]} columnas")
print(f"Clips únicos (id): {df_all['id'].nunique()}")
print(f"\nEstados:\n{df_all['estado'].value_counts()}")
print(f"\nSesiones:\n{df_all['sesion'].value_counts()}")


In [ ]:
clip_stats = df_all.groupby('id').agg(
    sesion      =('sesion',    'first'),
    n_frames    =('frame',     'count'),
    duracion_s  =('timestamp', 'max'),
    pct_resp    =('estado',    lambda x: (x=='respuesta').mean()*100),
    conf_media  =('confidence','mean'),
    success_rate=('success',   'mean'),
).reset_index()

print("Estadísticas por clip:")
print(clip_stats.describe().round(3))


---
## 1. Calidad del seguimiento

In [ ]:
clip_conf = df_all.groupby('id').agg(conf_media=('confidence','mean'),
                                      sesion=('sesion','first')).sort_values('conf_media')
clip_suc  = df_all.groupby('id')['success'].mean() * 100
mu_c      = df_all['confidence'].mean()

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

axes[0].hist(df_all['confidence'].dropna(), bins=40,
             color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(mu_c,  color='red',    lw=2, linestyle='--', label=f'Media={mu_c:.3f}')
axes[0].axvline(0.85,  color='orange', lw=1.5, linestyle=':', label='Umbral 0.85')
axes[0].set_xlabel('Confidence'); axes[0].set_ylabel('Fotogramas')
axes[0].set_title('Distribución global de confidence')
axes[0].legend()

colors_b = ['#e07040' if s == 2 else '#4C72B0' for s in clip_conf['sesion']]
axes[1].barh(range(len(clip_conf)), clip_conf['conf_media'], color=colors_b, alpha=0.85)
axes[1].axvline(0.85, color='orange', lw=1.5, linestyle='--', label='0.85')
axes[1].set_yticks(range(len(clip_conf)))
axes[1].set_yticklabels(clip_conf.index, fontsize=6)
axes[1].set_xlabel('Confidence media'); axes[1].set_ylabel('Clip ID')
axes[1].set_title('Confidence media por clip')
s1_p = mpatches.Patch(color='#4C72B0', label='Sesión 1')
s2_p = mpatches.Patch(color='#e07040', label='Sesión 2')
axes[1].legend(handles=[s1_p, s2_p])

c_suc = ['#d73027' if v < 90 else '#4575b4' for v in clip_suc.sort_values()]
axes[2].barh(range(len(clip_suc)), clip_suc.sort_values().values, color=c_suc, alpha=0.85)
axes[2].axvline(90, color='orange', lw=1.5, linestyle='--', label='Umbral 90%')
axes[2].set_yticks(range(len(clip_suc)))
axes[2].set_yticklabels(clip_suc.sort_values().index, fontsize=6)
axes[2].set_xlabel('Success (%)'); axes[2].set_ylabel('Clip ID')
axes[2].set_title('Tasa de success por clip')
axes[2].legend()

clips_bajo_conf = clip_conf[clip_conf['conf_media'] < 0.85].index.tolist()
print(f"Clips con confidence media < 0.85: {clips_bajo_conf}")

plt.suptitle('Calidad del seguimiento facial (OpenFace)', fontweight='bold')
plt.tight_layout(); savefig('fig01_calidad_seguimiento.png'); plt.show()


In [ ]:
clip_conf_sorted = df_all.groupby('id')['confidence'].mean().sort_values()
worst3 = clip_conf_sorted.head(3).index.tolist()
best3  = clip_conf_sorted.tail(3).index.tolist()

for tag, clip_list, prefix in [
    ('Peor seguimiento', worst3, 'fig02a'),
    ('Mejor seguimiento', best3, 'fig02b')
]:

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

    for ax, cid in zip(axes, clip_list):
        sub  = df_all[df_all['id'] == cid]
        resp = sub[sub['estado'] == 'respuesta']['timestamp']

        ax.fill_between(
            sub['timestamp'],
            sub['confidence'],
            alpha=0.5,
            color='steelblue'
        )

        ax.axhline(0.85, color='orange', lw=1.3, linestyle='--')

        if len(resp):
            ax.axvspan(resp.min(), resp.max(), alpha=0.15, color='#DD8452')

        ax.set_title(f"Clip {cid} (μ={clip_conf_sorted[cid]:.3f})")
        ax.set_xlabel('Tiempo (s)')
        ax.set_ylim(0.45, 1.05)

    axes[0].set_ylabel('Confidence')

    plt.suptitle(f'Confidence temporal — {tag}', fontweight='bold')
    plt.tight_layout()

    savefig(f'{prefix}.png')
    plt.show()

In [ ]:
df_feat = df_all[(df_all['success']==1) & (df_all['confidence']>=0.80)].copy()
print(f"df_feat: {len(df_feat):,} fotogramas tras filtrar por calidad "
      f"(success=1, confidence≥0.80) — {len(df_feat)/len(df_all)*100:.1f}% del total")


---
## 2. Análisis descriptivo de las Unidades de Acción
### 2.1 Perfil basal de activación y estructura de correlación

In [ ]:
order = df_all[AU_COLS_R].mean().sort_values(ascending=False).index.tolist()
au_long = df_all[AU_COLS_R].melt(var_name='AU', value_name='Intensidad')

fig, ax = plt.subplots(figsize=(16, 6))
sns.violinplot(data=au_long[au_long['AU'].isin(order)],
               x='AU', y='Intensidad', order=order,
               palette='Blues_d', inner='quartile', ax=ax,
               density_norm='width', cut=0)
ax.set_xticklabels([a.replace('_r','') for a in order], rotation=45, ha='right')
means = df_all[AU_COLS_R].mean()
for xi, au in enumerate(order):
    ax.plot(xi, means[au], 'ro', ms=5, zorder=5)
ax.set_xlabel('Unidad de Acción'); ax.set_ylabel('Intensidad (OpenFace continua)')
ax.set_title('Distribución de intensidades por AU')
plt.tight_layout(); savefig('fig03_au_violinplot_global.png'); plt.show()

print("Medias globales por AU (ordenadas):")
print(df_all[AU_COLS_R].mean().sort_values(ascending=False).round(4).to_string())


In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import matplotlib.cm as cm

au_c_present = [c for c in AU_COLS_C_ALL if c in df_all.columns]
au_c_means   = df_all[au_c_present].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(13, 5))

norm = Normalize(
    vmin=au_c_means.min(),
    vmax=au_c_means.max()
)

cmap = cm.get_cmap('Oranges')
colors = cmap(norm(au_c_means.values))

bars = ax.bar(
    au_c_means.index,
    au_c_means.values,
    color=colors,
    edgecolor='white'
)

for bar, val in zip(bars, au_c_means.values):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.5,
        f'{val:.1f}%',
        ha='center',
        va='bottom',
        fontsize=7
    )

ax.set_xticks(range(len(au_c_means)))
ax.set_xticklabels(
    [c.replace('_c', '') for c in au_c_means.index],
    rotation=45,
    ha='right'
)

ax.set_ylabel('Frecuencia de activación (%)')
ax.set_xlabel('AU (binaria)')
ax.set_title('Frecuencia de activación de AUs binarias por fotograma')

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label('Frecuencia de activación (%)')

plt.tight_layout()
savefig('fig04_au_binarias_frecuencia.png')
plt.show()

In [ ]:
au_by_clip = df_all.groupby('id')[AU_COLS_R].mean()

sesion_map = df_all.groupby('id')['sesion'].first()

orden_df = pd.DataFrame({
    'id': au_by_clip.index,
    'sesion': sesion_map.loc[au_by_clip.index].values
})

orden_df['id_num'] = pd.to_numeric(orden_df['id'])

orden_ids = (
    orden_df
    .sort_values(['sesion', 'id_num'])
    ['id']
)

au_by_clip_sorted = au_by_clip.loc[orden_ids]

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(
    au_by_clip_sorted.T,
    cmap='YlOrRd',
    linewidths=0.2,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Intensidad media AU'},
    yticklabels=[AU_NAMES[a] for a in AU_COLS_R]
)

ax.set_xlabel('Clip ID')
ax.set_ylabel('Unidad de Acción')
ax.set_title('Heatmap intensidad media de AUs por clip')
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)

n_ses1 = (sesion_map == 1).sum()
ax.axvline(n_ses1, color='black', lw=2, linestyle='--')
ax.text(n_ses1-15, -0.8, 'Sesión 1', ha='right', va='top', fontsize=8)
ax.text(n_ses1+10, -0.8, 'Sesión 2', ha='left', va='top', fontsize=8)

plt.tight_layout()
savefig('fig05_heatmap_au_por_clip.png')
plt.show()

In [ ]:
corr_au = df_all[AU_COLS_R].corr()
labels  = [AU_NAMES[a] for a in AU_COLS_R]
mask    = np.triu(np.ones_like(corr_au, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_au, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.4, annot=True, fmt='.2f',
            annot_kws={'size': 7}, xticklabels=labels, yticklabels=labels,
            ax=ax, cbar_kws={'shrink': 0.65})
ax.set_title('Matriz de correlación entre intensidades de AUs')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7)

high_corr = [(AU_COLS_R[i].replace('_r',''), AU_COLS_R[j].replace('_r',''), corr_au.iloc[i,j])
             for i in range(len(AU_COLS_R)) for j in range(i)
             if abs(corr_au.iloc[i,j]) > 0.5]
print("Pares con |r| > 0.5:")
for a, b, r in sorted(high_corr, key=lambda x: -abs(x[2])):
    print(f"  {a} – {b}: r = {r:.3f}")

plt.tight_layout(); savefig('fig06_correlacion_aus.png'); plt.show()


### 2.2 Estabilidad temporal y consistencia entre clips

In [ ]:
au_var_by_clip = df_all.groupby('id')[AU_COLS_R].std().mean(axis=1)
clip_most_active = au_var_by_clip.idxmax()
print(f"Clip más activo: {clip_most_active}  (σ_media={au_var_by_clip[clip_most_active]:.4f})")

sub = df_all[df_all['id']==clip_most_active].copy().sort_values('timestamp')
top_aus = sub[AU_COLS_R].std().nlargest(6).index.tolist()


In [ ]:
fig, axes = plt.subplots(len(top_aus), 1, figsize=(14, 11), sharex=True)
resp = sub[sub['estado']=='respuesta']['timestamp']

for ax, au in zip(axes, top_aus):
    sig = sub[au].values
    sig_sm = savgol_filter(sig, window_length=min(9, len(sig)//4*2+1), polyorder=2) \
             if len(sig) > 15 else sig
    ax.plot(sub['timestamp'], sig,    lw=0.8, color='lightsteelblue', alpha=0.6)
    ax.plot(sub['timestamp'], sig_sm, lw=1.8, color='steelblue', label='Suavizado SG')
    ax.fill_between(sub['timestamp'], sig_sm, alpha=0.15, color='steelblue')
    if len(resp): ax.axvspan(resp.min(), resp.max(), alpha=0.18, color='#DD8452',
                              label='Respuesta')
    ax.set_ylabel(AU_NAMES[au], fontsize=8); ax.set_ylim(bottom=0)
    if ax is axes[0]: ax.legend(fontsize=7, ncol=2)

axes[-1].set_xlabel('Tiempo (s)')
fig.suptitle(f'Evolución temporal AUs más activas (Clip {clip_most_active})',
             fontweight='bold')
plt.tight_layout(); savefig('fig07_temporal_aus_clip.png'); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
for i, au in enumerate(AU_COLS_R):
    active = sub.loc[sub[au] > 0.5, 'timestamp']
    if len(active):
        ax.scatter(active, [i]*len(active), marker='|', s=60,
                   color='steelblue', alpha=0.7, linewidths=1.2)
resp = sub[sub['estado']=='respuesta']['timestamp']
if len(resp): ax.axvspan(resp.min(), resp.max(), alpha=0.12, color='#DD8452',
                          label='Ventana respuesta')
ax.set_yticks(range(len(AU_COLS_R)))
ax.set_yticklabels([AU_NAMES[a] for a in AU_COLS_R], fontsize=8)
ax.set_xlabel('Tiempo (s)')
ax.set_title(f'Rasterplot de activación de AUs (umbral > 0.5) — Clip {clip_most_active}')
ax.legend()
plt.tight_layout(); savefig('fig08_rasterplot_aus.png'); plt.show()


In [ ]:
clip_ids_ordered = df_all['id'].unique()
N_BINS = 50

temporal_matrix = []
clip_labels_tm  = []
for cid in clip_ids_ordered:
    sub_c = df_all[df_all['id']==cid].sort_values('timestamp')
    dur   = sub_c['timestamp'].max()
    if dur < 1: continue
    sub_c['bin'] = pd.cut(sub_c['timestamp'], bins=N_BINS, labels=False)
    row = sub_c.groupby('bin')[AU_COLS_R].mean().reindex(range(N_BINS))
    temporal_matrix.append(row.values.T)
    clip_labels_tm.append(cid)

mean_temporal = np.nanmean(np.array(temporal_matrix), axis=0)
t_axis = np.linspace(0, 100, N_BINS)

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(mean_temporal, cmap='YlOrRd', ax=ax,
            xticklabels=np.round(np.linspace(0,100,11),0).astype(int),
            yticklabels=[AU_NAMES[a] for a in AU_COLS_R],
            cbar_kws={'label': 'Intensidad media AU'})
ax.set_xlabel('Tiempo normalizado del clip (%)')
ax.set_ylabel('AU')
ax.set_title('Evolución temporal media de todas las AUs')
ax.set_xticks(np.linspace(0, N_BINS, 11))
plt.tight_layout(); savefig('fig09_heatmap_temporal_aus.png'); plt.show()


In [ ]:
au_stats_clip = df_feat.groupby('id')[AU_COLS_R].agg(['mean','std'])
au_means_list = [au_stats_clip[au]['mean'].values for au in AU_COLS_R]

fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot(au_means_list,
                labels=[a.replace('_r','') for a in AU_COLS_R],
                patch_artist=True, notch=False,
                medianprops=dict(color='black', lw=2))
pal_b = sns.color_palette('Blues_d', len(AU_COLS_R))
for patch, color in zip(bp['boxes'], pal_b):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_xticklabels([a.replace('_r','') for a in AU_COLS_R], rotation=45, ha='right')
ax.set_ylabel('Intensidad media por clip')
ax.set_title('Variabilidad inter-clip de cada AU (distribución de medias por clip)')
plt.tight_layout(); savefig('fig34_variabilidad_interclip.png'); plt.show()

cv_interclip = {au.replace('_r',''): au_stats_clip[au]['mean'].std() /
                (au_stats_clip[au]['mean'].mean()+1e-9)
                for au in AU_COLS_R}
print("CV inter-clip (menor = más estable entre clips):")
for au, cv in sorted(cv_interclip.items(), key=lambda x: x[1]):
    print(f"  {au}: CV={cv:.3f}")


---
## 3. Movimiento involuntario de cabeza

In [ ]:
rot_cols  = ['pose_Rx', 'pose_Ry', 'pose_Rz']
rot_names = ['Pitch (cabeceo)', 'Yaw (giro lateral)', 'Roll (inclinación)']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, label in zip(axes, rot_cols, rot_names):
    data = df_all[col].dropna()
    mu, sigma = stats.norm.fit(data)
    xr = np.linspace(data.min(), data.max(), 300)
    p5, p95 = np.percentile(data, [5, 95])
    stat_ks, p_ks = stats.kstest(data, 'norm', args=(mu, sigma))

    ax.hist(data, bins=60, color='mediumpurple', edgecolor='white', alpha=0.8, density=True)
    ax.plot(xr, stats.norm.pdf(xr, mu, sigma), 'r--', lw=2,
            label=f'N(μ={mu:.3f}, σ={sigma:.3f})')
    ax.axvline(p5,  color='green', lw=1.2, linestyle=':', alpha=0.8)
    ax.axvline(p95, color='green', lw=1.2, linestyle=':', alpha=0.8,
               label=f'P5-P95: [{p5:.2f}, {p95:.2f}]')
    ax.set_xlabel('Ángulo (rad)'); ax.set_ylabel('Densidad')
    ax.set_title(f'{label}\n(KS p={p_ks:.3f})')
    ax.legend(fontsize=7)

plt.suptitle('Distribución de rotaciones de cabeza con ajuste gaussiano',
             fontweight='bold')
plt.tight_layout(); savefig('fig10_rotaciones_cabeza.png'); plt.show()


In [ ]:
def angular_velocity(group):
    g = group.sort_values('timestamp').copy()
    dt = g['timestamp'].diff().replace(0, np.nan)
    for c in ['pose_Rx','pose_Ry','pose_Rz']:
        g[f'vel_{c}'] = g[c].diff() / dt
    g['vel_total'] = np.sqrt(g['vel_pose_Rx']**2 +
                              g['vel_pose_Ry']**2 +
                              g['vel_pose_Rz']**2)
    return g

df_all = df_all.groupby('id', group_keys=False).apply(angular_velocity)

fig, ax = plt.subplots(figsize=(7, 6))
for estado, color in ESTADO_COLORS.items():
    se = df_all[df_all['estado']==estado]
    ax.scatter(se['pose_Ry'], se['pose_Rx'], s=3, alpha=0.25,
               color=color, label=estado.capitalize())
ax.set_xlabel('Yaw (rad)'); ax.set_ylabel('Pitch (rad)')
ax.set_title('Espacio YawxPitch (pregunta vs respuesta)')
ax.legend(markerscale=4)
plt.tight_layout(); savefig('fig11a_espacio_yaw_pitch.png'); plt.show()

sub2 = df_all[df_all['id']==clip_most_active].sort_values('timestamp').reset_index(drop=True)
fig, ax = plt.subplots(figsize=(7, 6))
cmap_v = plt.cm.viridis(np.linspace(0,1,len(sub2)))
for i in range(len(sub2)-1):
    ax.annotate('',
        xy=(sub2['pose_Ry'].iloc[i+1], sub2['pose_Rx'].iloc[i+1]),
        xytext=(sub2['pose_Ry'].iloc[i], sub2['pose_Rx'].iloc[i]),
        arrowprops=dict(arrowstyle='->', color=cmap_v[i], lw=0.7))
sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(0, sub2['timestamp'].max()))
plt.colorbar(sm, ax=ax, label='Tiempo (s)')
ax.set_xlabel('Yaw (rad)'); ax.set_ylabel('Pitch (rad)')
ax.set_title(f'Trayectoria temporal de cabeza (Clip {clip_most_active})')
plt.tight_layout(); savefig('fig11b_trayectoria_cabeza.png'); plt.show()

vel_preg = df_all[df_all['estado']=='pregunta']['vel_total'].dropna()
vel_resp = df_all[df_all['estado']=='respuesta']['vel_total'].dropna()
u_stat, p_val = stats.mannwhitneyu(vel_preg, vel_resp, alternative='two-sided')

fig, ax = plt.subplots(figsize=(7, 6))
bp = ax.boxplot([vel_preg, vel_resp], labels=['Pregunta','Respuesta'],
                patch_artist=True, notch=True,
                medianprops=dict(color='black', lw=2), showfliers=False)
for patch, color in zip(bp['boxes'], ESTADO_COLORS.values()):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.text(0.05, 0.95, f'Mann-Whitney U\np = {p_val:.4f}\n(no significativo)',
        transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
ax.set_ylabel('Velocidad angular (rad/s)')
ax.set_title('Velocidad angular de cabeza por estado')
plt.tight_layout(); savefig('fig11c_velocidad_angular.png'); plt.show()
print(f"Mann-Whitney U: estadístico={u_stat:.0f}, p={p_val:.4f}")


### 3.1 Análisis de mirada (gaze) — complementario

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for estado, color in ESTADO_COLORS.items():
    se = df_all[df_all['estado']==estado]
    ax.scatter(se['gaze_angle_x'], se['gaze_angle_y'],
               s=4, alpha=0.3, color=color, label=estado.capitalize())
for estado, color in ESTADO_COLORS.items():
    se = df_all[df_all['estado']==estado]
    mx, my = se['gaze_angle_x'].mean(), se['gaze_angle_y'].mean()
    sx, sy = se['gaze_angle_x'].std(), se['gaze_angle_y'].std()
    ell = plt.matplotlib.patches.Ellipse((mx,my), 2*sx, 2*sy,
                                          fill=False, edgecolor=color, lw=2, linestyle='--')
    ax.add_patch(ell)
ax.set_xlabel('Gaze X (rad)'); ax.set_ylabel('Gaze Y (rad)')
ax.set_title('Espacio de mirada (elipses = media±std)')
ax.legend(markerscale=4)
plt.tight_layout(); savefig('fig12a_gaze_scatter.png'); plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for estado, color in ESTADO_COLORS.items():
    data = df_all[df_all['estado']==estado]['gaze_angle_x'].dropna()
    sns.kdeplot(data, ax=ax, color=color, fill=True, alpha=0.35, label=estado.capitalize())
    ax.axvline(data.mean(), color=color, lw=1.5, linestyle='--', alpha=0.8)
gp = df_all[df_all['estado']=='pregunta']['gaze_angle_x'].dropna()
gr = df_all[df_all['estado']=='respuesta']['gaze_angle_x'].dropna()
_, p_mw = stats.mannwhitneyu(gp, gr, alternative='two-sided')
ax.set_title(f'Gaze horizontal — ángulo X (p={p_mw:.4f})')
ax.set_xlabel('Ángulo (rad)'); ax.set_ylabel('Densidad'); ax.legend(fontsize=8)
plt.tight_layout(); savefig('fig12b_gaze_horizontal.png'); plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for estado, color in ESTADO_COLORS.items():
    data = df_all[df_all['estado']==estado]['gaze_angle_y'].dropna()
    sns.kdeplot(data, ax=ax, color=color, fill=True, alpha=0.35, label=estado.capitalize())
    ax.axvline(data.mean(), color=color, lw=1.5, linestyle='--', alpha=0.8)
gp = df_all[df_all['estado']=='pregunta']['gaze_angle_y'].dropna()
gr = df_all[df_all['estado']=='respuesta']['gaze_angle_y'].dropna()
_, p_mw = stats.mannwhitneyu(gp, gr, alternative='two-sided')
ax.set_title(f'Gaze vertical — ángulo Y (p={p_mw:.4f})')
ax.set_xlabel('Ángulo (rad)'); ax.set_ylabel('Densidad'); ax.legend(fontsize=8)
plt.tight_layout(); savefig('fig12c_gaze_vertical.png'); plt.show()


---
## 4. Pose corporal: cobertura y análisis de brazos

In [ ]:
body_cols_all = [c for c in df_all.columns if any(k in c for k in ['Hombro','Codo','Muneca'])]
missing_pct   = df_all[body_cols_all].isna().mean()*100
missing_pct   = missing_pct.sort_values(ascending=True)

body_discard = missing_pct[missing_pct > 50].index.tolist()
body_ok      = [c for c in BODY_COLS_C if c in df_all.columns
                and df_all[c].isna().mean() < 0.5]
print(f"Variables descartadas (>50% NaN): {body_discard}")
print(f"Variables retenidas: {body_ok}")


In [ ]:
def limb_velocity(group, cx, cy, out):
    g  = group.sort_values('timestamp').copy()
    dt = g['timestamp'].diff().replace(0, np.nan)
    g[out] = np.sqrt(g[cx].diff()**2 + g[cy].diff()**2) / dt
    return g

for cx, cy, out in [('MunecaIzq_X_c','MunecaIzq_Y_c','vel_MunecaIzq'),
                    ('MunecaDer_X_c','MunecaDer_Y_c','vel_MunecaDer')]:
    if cx in df_all.columns and cy in df_all.columns:
        df_all = df_all.groupby('id', group_keys=False).apply(
            lambda g: limb_velocity(g, cx, cy, out))

sub_b  = df_all[df_all['id']==clip_most_active].sort_values('timestamp')
resp_b = sub_b[sub_b['estado']=='respuesta']['timestamp']

fig, ax = plt.subplots(figsize=(13, 4))
for col, color, lbl in [('MunecaIzq_Y_c','#4C72B0','Muñeca Izq.'),
                         ('MunecaDer_Y_c','#DD8452','Muñeca Der.')]:
    if col in sub_b.columns:
        ax.plot(sub_b['timestamp'], sub_b[col], lw=1.5, color=color, label=lbl)
if len(resp_b): ax.axvspan(resp_b.min(), resp_b.max(), alpha=0.14, color='#DD8452', label='Respuesta')
ax.set_ylabel('Coord. Y (norm.)'); ax.set_xlabel('Tiempo (s)')
ax.set_title('Posición vertical de muñecas')
ax.invert_yaxis(); ax.legend()
plt.tight_layout(); savefig('fig14a_muneca_posicion_y.png'); plt.show()

fig, ax = plt.subplots(figsize=(13, 4))
for col, color, lbl in [('MunecaIzq_X_c','#4C72B0','Muñeca Izq.'),
                         ('MunecaDer_X_c','#DD8452','Muñeca Der.')]:
    if col in sub_b.columns:
        ax.plot(sub_b['timestamp'], sub_b[col], lw=1.5, color=color, label=lbl)
if len(resp_b): ax.axvspan(resp_b.min(), resp_b.max(), alpha=0.14, color='#DD8452', label='Respuesta')
ax.set_ylabel('Coord. X (norm.)'); ax.set_xlabel('Tiempo (s)')
ax.set_title('Posición horizontal de muñecas')
ax.legend()
plt.tight_layout(); savefig('fig14b_muneca_posicion_x.png'); plt.show()

fig, ax = plt.subplots(figsize=(13, 4))
for vc, color, lbl in [('vel_MunecaIzq','#4C72B0','Vel. Izq.'),
                        ('vel_MunecaDer','#DD8452','Vel. Der.')]:
    if vc in sub_b.columns:
        ax.plot(sub_b['timestamp'],
                np.clip(sub_b[vc], 0, np.percentile(sub_b[vc].dropna(), 95)),
                lw=1.5, color=color, label=lbl)
if len(resp_b): ax.axvspan(resp_b.min(), resp_b.max(), alpha=0.14, color='#DD8452', label='Respuesta')
ax.set_ylabel('Velocidad (px_norm/s)'); ax.set_xlabel('Tiempo (s)')
ax.set_title('Velocidad de muñecas (recortada p95)')
ax.legend()
plt.tight_layout(); savefig('fig14c_muneca_velocidad.png'); plt.show()


In [ ]:
vel_cols_ok = [c for c in ['vel_MunecaIzq','vel_MunecaDer'] if c in df_all.columns]

for vc in vel_cols_ok:
    lbl   = 'Izquierda' if 'Izq' in vc else 'Derecha'
    fname = f"fig15{'a' if 'Izq' in vc else 'b'}_velocidad_muneca_{lbl.lower()}.png"
    vp = df_all[df_all['estado']=='pregunta'][vc].dropna()
    vr = df_all[df_all['estado']=='respuesta'][vc].dropna()
    u_s, p_v = stats.mannwhitneyu(vp, vr, alternative='two-sided')

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))

    bp = axes[0].boxplot([vp, vr], labels=['Pregunta','Respuesta'],
                          patch_artist=True, notch=True, showfliers=False,
                          medianprops=dict(color='black', lw=2))
    for patch, color in zip(bp['boxes'], ESTADO_COLORS.values()):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    axes[0].text(0.05, 0.95, f'p={p_v:.4f}', transform=axes[0].transAxes,
                 va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axes[0].set_title(f'Velocidad Muñeca {lbl} — boxplot')
    axes[0].set_ylabel('Velocidad')

    for estado, color in ESTADO_COLORS.items():
        data = np.clip(df_all[df_all['estado']==estado][vc].dropna(), 0,
                       np.percentile(df_all[vc].dropna(), 95))
        sns.kdeplot(data, ax=axes[1], color=color, fill=True, alpha=0.35,
                    label=estado.capitalize())
    axes[1].set_xlabel('Velocidad (p95)'); axes[1].set_title(f'Velocidad Muñeca {lbl} — KDE')
    axes[1].legend(fontsize=7)

    plt.suptitle(f'Velocidad Muñeca {lbl} por estado', fontweight='bold')
    plt.tight_layout(); savefig(fname); plt.show()


In [ ]:
if len(body_ok) > 0:
    corr_cross = df_feat[AU_COLS_R + body_ok].corr()
    corr_block = corr_cross.loc[AU_COLS_R, body_ok]

    fig, ax = plt.subplots(figsize=(max(9, len(body_ok)*0.75),
                                     max(7, len(AU_COLS_R)*0.45)))
    sns.heatmap(corr_block, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                annot=True, fmt='.2f', annot_kws={'size':7}, linewidths=0.3,
                xticklabels=[c.replace('_c','').replace('_','\n') for c in body_ok],
                yticklabels=[AU_NAMES[a] for a in AU_COLS_R],
                ax=ax, cbar_kws={'label':'Correlación Pearson'})
    ax.set_title('Correlación cruzada AU ↔ coordenadas corporales')
    plt.tight_layout(); savefig('fig33_correlacion_au_cuerpo.png'); plt.show()

    high = [(AU_COLS_R[i].replace('_r',''), body_ok[j], corr_block.iloc[i,j])
            for i in range(len(AU_COLS_R)) for j in range(len(body_ok))
            if abs(corr_block.iloc[i,j]) > 0.3]
    if high:
        print("Pares AU-cuerpo con |r|>0.3:")
        for a,b,r in sorted(high, key=lambda x: -abs(x[2])):
            print(f"  {a} - {b}: r={r:.3f}")
    else:
        print("No hay pares AU-cuerpo con |r|>0.3")


---
## 5. Comparativa estadística: pregunta vs. respuesta

In [ ]:
results = []
for au in AU_COLS_R:
    gp = df_all[df_all['estado']=='pregunta'][au].dropna()
    gr = df_all[df_all['estado']=='respuesta'][au].dropna()
    u_stat, p_val = stats.mannwhitneyu(gp, gr, alternative='two-sided')
    n1, n2 = len(gp), len(gr)
    r_eff = 1 - (2*u_stat)/(n1*n2)
    pooled_std = np.sqrt((gp.var()+gr.var())/2)
    cohens_d   = (gr.mean()-gp.mean()) / (pooled_std+1e-9)
    results.append({
        'AU':              au.replace('_r',''),
        'Descripcion':     AU_NAMES[au],
        'Media_pregunta':  round(gp.mean(),4),
        'Media_respuesta': round(gr.mean(),4),
        'Delta':           round(gr.mean()-gp.mean(),4),
        'U_stat':          u_stat,
        'p_valor':         p_val,
        'p_bonferroni':    min(p_val*len(AU_COLS_R), 1.0),
        'r_biserial':      round(r_eff,4),
        'cohens_d':        round(cohens_d,4),
    })

df_stats = pd.DataFrame(results).sort_values('p_bonferroni')
print("Tests Mann-Whitney U + corrección Bonferroni:")
print(df_stats[['AU','Media_pregunta','Media_respuesta','Delta',
                'p_valor','p_bonferroni','r_biserial','cohens_d']].to_string(
    index=False, float_format='%.5f'))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

diff_sorted = df_stats.sort_values('Delta')
colors_d = ['#4575b4' if v>0 else '#d73027' for v in diff_sorted['Delta']]
axes[0].barh(diff_sorted['AU'], diff_sorted['Delta'], color=colors_d, alpha=0.85)
axes[0].axvline(0, color='black', lw=1)
axes[0].set_xlabel('Δ Intensidad media (Respuesta - Pregunta)')
axes[0].set_title('A — Diferencia de activación de AUs entre estados')
p_p = mpatches.Patch(color='#4575b4', label='↑ en Respuesta')
n_p = mpatches.Patch(color='#d73027', label='↑ en Pregunta')
axes[0].legend(handles=[p_p, n_p])

sig_mask = df_stats['p_bonferroni'] < 0.05
colors_v  = ['#d73027' if s else '#aaaaaa' for s in sig_mask]
axes[1].scatter(df_stats['r_biserial'],
                -np.log10(df_stats['p_bonferroni'].clip(lower=1e-10)),
                c=colors_v, s=60, alpha=0.85, edgecolors='white', linewidths=0.5)
axes[1].axhline(-np.log10(0.05), color='orange', lw=1.5, linestyle='--',
                label='p=0.05 (Bonferroni)')
axes[1].axvline(0.1,  color='green', lw=1, linestyle=':', alpha=0.6)
axes[1].axvline(-0.1, color='green', lw=1, linestyle=':', alpha=0.6)
for _, row in df_stats[sig_mask].iterrows():
    axes[1].annotate(row['AU'],
        (row['r_biserial'], -np.log10(row['p_bonferroni']+1e-10)),
        textcoords='offset points', xytext=(4,4), fontsize=8)
axes[1].set_xlabel('Effect size r (biserial de rango)')
axes[1].set_ylabel('-log10(p Bonferroni)')
axes[1].set_title('B — Volcano plot: significación vs tamaño del efecto')
axes[1].legend()

plt.suptitle('Tests estadísticos AU: pregunta vs respuesta', fontweight='bold')
plt.tight_layout(); savefig('fig16_tests_estadisticos_au.png'); plt.show()


In [ ]:
sig_aus_r = [f"{row['AU']}_r" for _, row in df_stats.iterrows()
             if row['p_bonferroni'] < 0.05 and f"{row['AU']}_r" in df_all.columns]
print(f"AUs significativas (p_Bonf<0.05): {[a.replace('_r','') for a in sig_aus_r]}")

if sig_aus_r:
    group1 = sig_aus_r[:2]
    fig, axes_arr = plt.subplots(1, len(group1), figsize=(6*len(group1), 5))
    if len(group1) == 1: axes_arr = [axes_arr]
    for ax, au in zip(axes_arr, group1):
        for estado, color in ESTADO_COLORS.items():
            data = df_all[df_all['estado']==estado][au].dropna()
            sns.kdeplot(data, ax=ax, color=color, fill=True, alpha=0.35,
                        label=f"{estado.capitalize()} (μ={data.mean():.3f})")
            ax.axvline(data.mean(), color=color, lw=1.5, linestyle='--')
        row_s = df_stats[df_stats['AU']==au.replace('_r','')].iloc[0]
        ax.set_title(f"{au.replace('_r','')} — {AU_NAMES[au]}\n"
                     f"p_Bonf={row_s['p_bonferroni']:.5f}, r={row_s['r_biserial']:.4f}")
        ax.set_xlabel('Intensidad'); ax.legend(fontsize=7)
    plt.suptitle('Distribuciones de AUs significativas (I)', fontweight='bold')
    plt.tight_layout(); savefig('fig17a_aus_sig_grupo1.png'); plt.show()

    group2 = sig_aus_r[2:5]
    if group2:
        nc2 = len(group2)
        fig, axes_arr2 = plt.subplots(1, nc2, figsize=(5*nc2, 5))
        if nc2 == 1: axes_arr2 = [axes_arr2]
        for ax, au in zip(axes_arr2, group2):
            for estado, color in ESTADO_COLORS.items():
                data = df_all[df_all['estado']==estado][au].dropna()
                sns.kdeplot(data, ax=ax, color=color, fill=True, alpha=0.35,
                            label=f"{estado.capitalize()} (μ={data.mean():.3f})")
                ax.axvline(data.mean(), color=color, lw=1.5, linestyle='--')
            row_s = df_stats[df_stats['AU']==au.replace('_r','')].iloc[0]
            ax.set_title(f"{au.replace('_r','')} — {AU_NAMES[au]}\n"
                         f"p_Bonf={row_s['p_bonferroni']:.5f}, r={row_s['r_biserial']:.4f}")
            ax.set_xlabel('Intensidad'); ax.legend(fontsize=7)
        plt.suptitle('Distribuciones de AUs significativas (II)', fontweight='bold')
        plt.tight_layout(); savefig('fig17b_aus_sig_grupo2.png'); plt.show()


---
## 6. Índice de viabilidad comunicativa (SNR)

In [ ]:
snr_data = []
for au in AU_COLS_R:
    mp = df_feat[df_feat['estado']=='pregunta'][au]
    mr = df_feat[df_feat['estado']=='respuesta'][au]
    mu_p, mu_r = mp.mean(), mr.mean()
    std_p = mp.std()
    pooled = np.sqrt((mp.var()+mr.var())/2)
    snr_data.append({
        'AU':         au.replace('_r',''),
        'Descripcion':AU_NAMES[au],
        'SNR':        (mu_r-mu_p)/(std_p+1e-9),
        'cohens_d':   (mu_r-mu_p)/(pooled+1e-9),
        'mu_resp':    mu_r, 'mu_preg': mu_p, 'std_preg': std_p,
    })
df_snr = pd.DataFrame(snr_data).sort_values('SNR', ascending=False)

print("SNR y Cohen's d por AU:")
print(df_snr[['AU','Descripcion','SNR','cohens_d','mu_resp','mu_preg']].to_string(
    index=False, float_format='%.4f'))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors_snr = ['#2166ac' if v > 0.2 else '#d73027' if v < -0.2 else '#aaaaaa'
              for v in df_snr['SNR']]

ax.barh(df_snr['AU'], df_snr['SNR'],
        color=colors_snr, alpha=0.85, edgecolor='white')

ax.axvline(0.2, color='green', lw=1.5, linestyle='--',
           alpha=0.7, label='SNR=±0.2')
ax.axvline(-0.2, color='green', lw=1.5, linestyle='--', alpha=0.7)
ax.axvline(0, color='black', lw=1)

ax.set_xlabel('SNR (Δμ / σ_pregunta)')
ax.set_title('Índice señal/ruido por AU')

b_p = mpatches.Patch(color='#2166ac', label='SNR > 0.2 (candidata)')
r_p = mpatches.Patch(color='#d73027', label='SNR < -0.2 (suprimida)')
g_p = mpatches.Patch(color='#aaaaaa', label='No discriminativa')

ax.legend(handles=[b_p, r_p, g_p], loc='lower left')

plt.tight_layout()
savefig('fig30A_snr.png')
plt.show()

In [ ]:
cv_data = []
for au in AU_COLS_R:
    for estado in ['pregunta','respuesta']:
        v = df_feat[df_feat['estado']==estado][au]
        cv_data.append({'AU': au.replace('_r',''), 'Estado': estado.capitalize(),
                        'CV': v.std()/(v.mean()+1e-9)})
df_cv = pd.DataFrame(cv_data).pivot(index='AU', columns='Estado', values='CV')

fig, ax = plt.subplots(figsize=(13, 6))
x_pos = np.arange(len(df_cv)); w = 0.35
for i, (col, color) in enumerate(zip(df_cv.columns, ESTADO_COLORS.values())):
    ax.bar(x_pos+i*w, df_cv[col], w, label=col, color=color, alpha=0.8, edgecolor='white')
ax.set_xticks(x_pos+w/2)
ax.set_xticklabels(df_cv.index, rotation=45, ha='right')
ax.set_ylabel('CV = σ/μ  (menor = más consistente)')
ax.set_title('Coeficiente de variación de AUs por estado')
ax.legend(title='Estado')
plt.tight_layout(); savefig('fig31_cv_aus.png'); plt.show()

candidatas_r = [f"{a}_r" for a in df_snr[df_snr['SNR']>0.08]['AU'].tolist()
                if f"{a}_r" in df_feat.columns][:6]
if candidatas_r:
    nc2 = min(3, len(candidatas_r))
    nr2 = (len(candidatas_r)+nc2-1)//nc2
    fig, axarr = plt.subplots(nr2, nc2, figsize=(5*nc2, 4*nr2))
    axarr = np.array(axarr).flatten() if nr2*nc2>1 else [axarr]
    for ax, au in zip(axarr, candidatas_r):
        for estado, color in ESTADO_COLORS.items():
            d = df_feat[df_feat['estado']==estado][au].dropna()
            sns.kdeplot(d, ax=ax, color=color, fill=True, alpha=0.35,
                        label=f"{estado.capitalize()} μ={d.mean():.3f}")
            ax.axvline(d.mean(), color=color, lw=1.5, linestyle='--')
        snr_val = df_snr[df_snr['AU']==au.replace('_r','')]['SNR'].values[0]
        ax.set_title(f"{au.replace('_r','')} — {AU_NAMES[au]}\nSNR={snr_val:.3f}")
        ax.set_xlabel('Intensidad'); ax.legend(fontsize=7)
    for ax in axarr[len(candidatas_r):]: ax.set_visible(False)
    fig.suptitle('Distribuciones de AUs candidatas (SNR>0.08)', fontweight='bold')
    plt.tight_layout(); savefig('fig32_kde_candidatas.png'); plt.show()


---
## 7. Reducción de dimensionalidad

In [ ]:
FEATURE_COLS = AU_COLS_R + POSE_HEAD + GAZE_COLS + body_ok

X_raw   = df_feat[FEATURE_COLS].values

imp    = SimpleImputer(strategy='median')
X_imp  = imp.fit_transform(X_raw)
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X_imp)

print(f"Matriz: {X_sc.shape[0]:,} fotogramas x {X_sc.shape[1]} features")
print(f"Features: {FEATURE_COLS}")


### 7.1 Análisis de Componentes Principales (PCA)

In [ ]:
pca     = PCA(n_components=min(15, X_sc.shape[1]), random_state=SEED)
X_pca   = pca.fit_transform(X_sc)
var_exp = pca.explained_variance_ratio_
cum_var = np.cumsum(var_exp)
feat_short = [f.replace('_r','').replace('pose_','p_').replace('gaze_angle_','g_')
              .replace('_c','') for f in FEATURE_COLS]
n80 = np.searchsorted(cum_var, 0.80) + 1
print(f"Varianza: 5 CPs={cum_var[4]*100:.1f}%  | 10 CPs={cum_var[9]*100:.1f}%  | {n80} CPs→80%")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax  = axes[0]
ax2 = ax.twinx()
ax.bar(range(1, len(var_exp)+1), var_exp*100, alpha=0.7, color='steelblue', label='Individual')
ax2.plot(range(1, len(cum_var)+1), cum_var*100, 'ro-', lw=2, ms=5, label='Acumulada')
ax2.axhline(80, color='orange', lw=1.5, linestyle='--', alpha=0.7)
ax2.axvline(n80, color='green', lw=1.2, linestyle=':')
ax.set_xlabel('Componente Principal')
ax.set_ylabel('Varianza individual (%)')
ax2.set_ylabel('Varianza acumulada (%)')
ax.set_title(f'Varianza explicada (primeras 5 CPs: {cum_var[4]*100:.1f}%)')
lines1 = ax.get_legend_handles_labels()
lines2 = ax2.get_legend_handles_labels()
ax.legend(
    lines1[0] + lines2[0],
    lines1[1] + lines2[1],
    fontsize=7,
    loc='upper right',
    bbox_to_anchor=(0.6, 1)
)

loadings = pca.components_[:2].T
top_idx  = np.argsort(np.abs(loadings[:,0]))[-12:]
ax3 = axes[1]
for i in top_idx:
    ax3.arrow(0, 0, loadings[i,0], loadings[i,1],
              head_width=0.02, head_length=0.01, fc='steelblue', ec='steelblue', alpha=0.75)
    ax3.text(loadings[i,0]*1.12, loadings[i,1]*1.12,
             feat_short[i], fontsize=7, ha='center', va='center')
ax3.set_xlim(-1.2, 1.2); ax3.set_ylim(-1.2, 1.2)
ax3.axhline(0, color='black', lw=0.7, alpha=0.4)
ax3.axvline(0, color='black', lw=0.7, alpha=0.4)
ax3.set_title('Biplot PCA — top 12 features en PC1')
ax3.set_xlabel(f'Carga PC1 ({var_exp[0]*100:.1f}%)')
ax3.set_ylabel(f'Carga PC2 ({var_exp[1]*100:.1f}%)')

plt.suptitle('Análisis de Componentes Principales (PCA)', fontweight='bold')
plt.tight_layout(); savefig('fig18_pca_varianza_biplot.png'); plt.show()


### 7.2 Proyecciones t-SNE

In [ ]:
N_PCS  = min(10, X_pca.shape[1])
n_tsne = min(3000, len(X_pca))
idx_t  = np.random.choice(len(X_pca), n_tsne, replace=False)

tsne = TSNE(
    n_components=2,
    perplexity=40,
    learning_rate=200,
    max_iter=1000,
    random_state=SEED,
    init='pca'
)

X_tsne = tsne.fit_transform(X_pca[idx_t, :N_PCS])

est_t  = df_feat['estado'].iloc[idx_t].values
ses_t  = df_feat['sesion'].iloc[idx_t].values
au14_t = df_feat['AU14_r'].iloc[idx_t].values

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

sc = axes[0].scatter(
    X_tsne[:, 0], X_tsne[:, 1],
    s=6, alpha=0.45,
    c=au14_t, cmap='YlOrRd'
)
plt.colorbar(sc, ax=axes[0], label='AU14 — hoyuelo')
axes[0].set_title('t-SNE: intensidad AU14')
axes[0].set_xlabel('Dim 1')
axes[0].set_ylabel('Dim 2')

for ses, color, lbl in [(1, '#2166ac', 'Sesión 1'),
                        (2, '#e07040', 'Sesión 2')]:
    m = ses_t == ses
    axes[1].scatter(
        X_tsne[m, 0], X_tsne[m, 1],
        s=6, alpha=0.45,
        color=color, label=lbl
    )

axes[1].set_title('t-SNE: sesión de grabación')
axes[1].set_xlabel('Dim 1')
axes[1].set_ylabel('Dim 2')
axes[1].legend(markerscale=3)

plt.suptitle(f't-SNE (n={n_tsne:,} fotogramas, perplexity=40)', fontweight='bold')
plt.tight_layout()
savefig('fig19_tsne.png')
plt.show()

---
## 8. Análisis de *clustering*: catálogo gestual
### 8.1 Método y selección del número de clusters

In [ ]:
N_PCS_C   = min(10, X_pca.shape[1])
X_clust   = X_pca[:, :N_PCS_C]
k_range   = range(2, 11)
inertias, sil_sc, ch_sc, db_sc = [], [], [], []

for k in k_range:
    km  = KMeans(n_clusters=k, random_state=SEED, n_init=10, max_iter=300)
    lbs = km.fit_predict(X_clust)
    inertias.append(km.inertia_)
    sil_sc.append(silhouette_score(X_clust, lbs, sample_size=5000, random_state=SEED))
    ch_sc.append(calinski_harabasz_score(X_clust, lbs))
    db_sc.append(davies_bouldin_score(X_clust, lbs))
    print(f"k={k}: Sil={sil_sc[-1]:.4f}, CH={ch_sc[-1]:.1f}, DB={db_sc[-1]:.4f}")

best_k_sil = list(k_range)[np.argmax(sil_sc)]
best_k_ch  = list(k_range)[np.argmax(ch_sc)]
best_k_db  = list(k_range)[np.argmin(db_sc)]
print(f"\nÓptimos: Sil→k={best_k_sil}  CH→k={best_k_ch}  DB→k={best_k_db}")
K_OPT = best_k_sil
print(f"K seleccionado: {K_OPT}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_range, inertias, 'bo-', lw=2, ms=7)
ax.set_xlabel('Número de clusters k'); ax.set_ylabel('Inercia (WCSS)')
ax.set_title('Método del codo (Elbow)'); ax.set_xticks(list(k_range))
plt.tight_layout(); savefig('fig21a_elbow.png'); plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_range, sil_sc, 'go-', lw=2, ms=7)
ax.axvline(best_k_sil, color='red', lw=1.5, linestyle='--', label=f'Óptimo k={best_k_sil}')
ax.set_xlabel('Número de clusters k'); ax.set_ylabel('Coeficiente de Silhouette')
ax.set_title('Silhouette (↑ mejor)'); ax.legend(); ax.set_xticks(list(k_range))
plt.tight_layout(); savefig('fig21b_silhouette.png'); plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_range, ch_sc, 'ro-', lw=2, ms=7)
ax.axvline(best_k_ch, color='purple', lw=1.5, linestyle='--', label=f'Óptimo k={best_k_ch}')
ax.set_xlabel('Número de clusters k'); ax.set_ylabel('Índice Calinski-Harabász')
ax.set_title('Calinski-Harabász (↑ mejor)'); ax.legend(); ax.set_xticks(list(k_range))
plt.tight_layout(); savefig('fig21c_calinski_harabasz.png'); plt.show()

K_OPT = best_k_sil
print(f"K seleccionado: {K_OPT}  (Sil={sil_sc[K_OPT-2]:.4f})")


In [ ]:
km_opt        = KMeans(n_clusters=K_OPT, random_state=SEED, n_init=10, max_iter=300)
cluster_labels = km_opt.fit_predict(X_clust)
df_feat        = df_feat.copy()
df_feat['cluster_km'] = cluster_labels
sil_vals = silhouette_samples(X_clust, cluster_labels)

fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10
pal_c   = [CLUSTER_COLORS[i % len(CLUSTER_COLORS)] for i in range(K_OPT)]
for i in range(K_OPT):
    ith  = np.sort(sil_vals[cluster_labels==i])
    y_up = y_lower + len(ith)
    ax.fill_betweenx(np.arange(y_lower, y_up), 0, ith,
                     facecolor=pal_c[i], alpha=0.7)
    ax.text(-0.07, y_lower + 0.5*len(ith), f'C{i}  (n={len(ith):,})', fontsize=9)
    y_lower = y_up + 10
ax.axvline(sil_vals.mean(), color='red', lw=1.8, linestyle='--',
           label=f'Media global: {sil_vals.mean():.4f}')
ax.set_xlabel('Coeficiente de Silhouette')
ax.set_title(f'Silhouette por cluster (K={K_OPT})')
ax.set_yticks([]); ax.legend()
plt.tight_layout(); savefig('fig22_silhouette.png'); plt.show()


In [ ]:
lbs_km = km_opt.predict(X_pca[idx_t, :N_PCS_C])

fig, ax = plt.subplots(figsize=(8, 7))
ul = np.unique(lbs_km)
cp = sns.color_palette('tab10', max(len(ul), 2))
for idx_l, lab in enumerate(ul):
    m = lbs_km == lab
    ax.scatter(X_tsne[m,0], X_tsne[m,1], s=6, alpha=0.5,
               color=cp[idx_l % len(cp)], label=f'Cluster {lab}')
ax.set_title(f'K-Means (k={K_OPT}) en espacio t-SNE')
ax.legend(markerscale=3, fontsize=8)
ax.set_xlabel('t-SNE Dim 1'); ax.set_ylabel('t-SNE Dim 2')
valid = lbs_km != -1
if valid.sum() > 1 and len(np.unique(lbs_km[valid])) > 1:
    s = silhouette_score(X_tsne[valid], lbs_km[valid])
    ax.text(0.02, 0.02, f'Silhouette={s:.4f}', transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
plt.tight_layout(); savefig('fig23_kmeans_tsne.png'); plt.show()


### 8.2 Descripción del catálogo gestual

In [ ]:
au_means_clust = df_feat.groupby('cluster_km')[AU_COLS_R].mean()
clust_size     = df_feat['cluster_km'].value_counts().sort_index()
estado_comp    = df_feat.groupby('cluster_km')['estado'].value_counts(
                    normalize=True).unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(au_means_clust.T, cmap='YlOrRd', annot=True, fmt='.2f',
            annot_kws={'size': 8}, linewidths=0.4,
            xticklabels=[f'Cluster {i}\n(n={clust_size[i]:,})' for i in au_means_clust.index],
            yticklabels=[AU_NAMES[a] for a in AU_COLS_R],
            ax=ax, cbar_kws={'label': 'Intensidad media AU'})
ax.set_title('Perfil de intensidad de AUs por cluster')
plt.tight_layout(); savefig('fig25_heatmap_au_clusters.png'); plt.show()

resp_global = (df_feat['estado']=='respuesta').mean()*100
print(f"% respuesta global: {resp_global:.1f}%")
for i in range(K_OPT):
    pct = (df_feat[df_feat['cluster_km']==i]['estado']=='respuesta').mean()*100
    diff = pct - resp_global
    print(f"Cluster {i} (n={clust_size.get(i,0):,}): {pct:.1f}% respuesta "
          f"({'+'if diff>=0 else ''}{diff:.1f}% vs global)")


In [ ]:
au_r_norm = au_means_clust[AU_COLS_R].copy()
au_r_norm = (au_r_norm - au_r_norm.min()) / (au_r_norm.max() - au_r_norm.min() + 1e-9)

N_ang  = len(AU_COLS_R)
angles = np.linspace(0, 2*np.pi, N_ang, endpoint=False).tolist()
angles += angles[:1]

for i in range(K_OPT):
    vals = au_r_norm.iloc[i].tolist() + [au_r_norm.iloc[i].tolist()[0]]
    top3 = au_r_norm.iloc[i].nlargest(3).index.tolist()
    top3_str = ', '.join([a.replace('_r','') for a in top3])

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax.plot(angles, vals, 'o-', lw=2, color=CLUSTER_COLORS[i % len(CLUSTER_COLORS)])
    ax.fill(angles, vals, alpha=0.22, color=CLUSTER_COLORS[i % len(CLUSTER_COLORS)])
    ax.set_thetagrids(np.degrees(angles[:-1]),
                      [a.replace('_r','') for a in AU_COLS_R], fontsize=8)
    ax.set_ylim(0, 1); ax.set_yticks([0.25, 0.5, 0.75]); ax.set_yticklabels([])
    ax.set_title(f'Cluster {i}  (n={clust_size.get(i,0):,})\nTop: {top3_str}',
                 fontsize=10, pad=18)
    plt.tight_layout()
    savefig(f'fig26_radar_cluster{i}.png'); plt.show()


In [ ]:
sub_cl  = df_feat[df_feat['id']==clip_most_active].copy().sort_values('timestamp')
resp_cl = sub_cl[sub_cl['estado']=='respuesta']['timestamp']

if len(sub_cl) > 0:
    au_dom     = au_means_clust[AU_COLS_R].max().nlargest(4).index.tolist()
    colors_au  = sns.color_palette('Set2', len(au_dom))

    fig, ax = plt.subplots(figsize=(14, 5))
    for au_col, col in zip(au_dom, colors_au):
        ax.plot(sub_cl['timestamp'], sub_cl[au_col],
                lw=1.5, color=col, label=AU_NAMES[au_col])
    if len(resp_cl):
        ax.axvspan(resp_cl.min(), resp_cl.max(), alpha=0.13,
                   color='#DD8452', label='Ventana respuesta')
    ax.set_ylabel('Intensidad AU'); ax.set_xlabel('Tiempo (s)')
    ax.set_title(f'Evolución de AUs dominantes (Clip {clip_most_active})')
    ax.legend(ncol=2, fontsize=8)
    plt.tight_layout(); savefig('fig27_aus_dominantes_temporal.png'); plt.show()


---
## 9. Análisis de latencia y estructura temporal de la respuesta

In [ ]:
lat_data = []
for cid in df_feat['id'].unique():
    sc  = df_feat[df_feat['id']==cid].sort_values('timestamp')
    rr  = sc[sc['estado']=='respuesta']
    pp  = sc[sc['estado']=='pregunta']
    ses = sc['sesion'].iloc[0]
    if len(rr) > 0 and len(pp) > 0:
        lat_data.append({
            'clip_id':      cid,
            'sesion':       ses,
            'latencia_s':   rr['timestamp'].min() - pp['timestamp'].max(),
            'dur_resp_s':   rr['timestamp'].max() - rr['timestamp'].min(),
            'dur_preg_s':   pp['timestamp'].max() - pp['timestamp'].min(),
            'pct_respuesta': len(rr)/len(sc)*100,
            'n_frames_total': len(sc),
            'cluster_resp': rr['cluster_km'].mode()[0] if 'cluster_km' in rr.columns else np.nan,
        })

df_lat = pd.DataFrame(lat_data)
print(df_lat.describe().round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df_lat['dur_resp_s'].dropna(), bins=20,
        color='#DD8452', edgecolor='white', alpha=0.85)
ax.axvline(df_lat['dur_resp_s'].median(), color='red', lw=2, linestyle='--',
           label=f"Mediana={df_lat['dur_resp_s'].median():.2f} s")
ax.set_xlabel('Duración ventana de respuesta (s)'); ax.set_ylabel('Número de clips')
ax.set_title('Distribución de la duración de la ventana de respuesta')
ax.legend()
plt.tight_layout(); savefig('fig28a_duracion_respuesta_hist.png'); plt.show()

ds = df_lat.sort_values('dur_resp_s', ascending=False)
fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(range(len(ds)), ds['dur_resp_s'],
        color=['#e07040' if s==2 else '#4C72B0' for s in ds['sesion']],
        alpha=0.85, edgecolor='white')
ax.set_yticks(range(len(ds)))
ax.set_yticklabels(ds['clip_id'], fontsize=7)
ax.set_xlabel('Duración (s)')
ax.set_title('Duración de la ventana de respuesta por clip')
s1p = mpatches.Patch(color='#4C72B0', label='Sesión 1')
s2p = mpatches.Patch(color='#e07040', label='Sesión 2')
ax.legend(handles=[s1p, s2p])
plt.tight_layout(); savefig('fig28b_duracion_por_clip.png'); plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df_lat['pct_respuesta'], bins=20,
        color='#4C72B0', edgecolor='white', alpha=0.85)
ax.axvline(df_lat['pct_respuesta'].median(), color='red', lw=2, linestyle='--',
           label=f"Mediana={df_lat['pct_respuesta'].median():.1f}%")
ax.set_xlabel('% del clip en estado respuesta'); ax.set_ylabel('Número de clips')
ax.set_title('Proporción del clip en estado de respuesta')
ax.legend()
plt.tight_layout(); savefig('fig28c_pct_respuesta.png'); plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(df_lat['dur_preg_s'], df_lat['dur_resp_s'],
           s=55, alpha=0.75, color='steelblue', edgecolors='white')
if len(df_lat.dropna()) >= 3:
    sl, ic, r, pv, _ = stats.linregress(df_lat['dur_preg_s'], df_lat['dur_resp_s'])
    xl = np.linspace(df_lat['dur_preg_s'].min(), df_lat['dur_preg_s'].max(), 100)
    ax.plot(xl, sl*xl+ic, 'r--', lw=2, label=f'r={r:.2f}, p={pv:.3f}')
    ax.legend()
ax.set_xlabel('Duración pregunta (s)'); ax.set_ylabel('Duración respuesta (s)')
ax.set_title('Duración de la pregunta vs duración de la respuesta')
plt.tight_layout(); savefig('fig28d_pregunta_vs_respuesta.png'); plt.show()


### 9.1 Event-Locked Average

In [ ]:
WINDOW_BEFORE = 2.0; WINDOW_AFTER = 4.0; FPS = 30
au_resp_mean  = df_feat[df_feat['estado']=='respuesta'][AU_COLS_R].mean()
top3_au       = au_resp_mean.nlargest(3).index.tolist()

aligned = {au: [] for au in top3_au}
t_common = np.arange(-WINDOW_BEFORE, WINDOW_AFTER, 1/FPS)

for cid in df_feat['id'].unique():
    sc  = df_feat[df_feat['id']==cid].sort_values('timestamp')
    rr  = sc[sc['estado']=='respuesta']
    if len(rr) == 0: continue
    t0  = rr['timestamp'].min()
    sc['t_al'] = sc['timestamp'] - t0
    win = sc[(sc['t_al'] >= -WINDOW_BEFORE) & (sc['t_al'] <= WINDOW_AFTER)]
    if len(win) < 8: continue
    for au in top3_au:
        aligned[au].append(np.interp(t_common, win['t_al'].values, win[au].values))

colors_ev = ['#e41a1c','#377eb8','#4daf4a']
fig, ax = plt.subplots(figsize=(12, 6))
for au, color in zip(top3_au, colors_ev):
    if not aligned[au]: continue
    mat  = np.array(aligned[au])
    mn   = np.nanmean(mat, axis=0)
    sem  = stats.sem(mat, axis=0, nan_policy='omit')
    ax.plot(t_common, mn, lw=2.5, color=color, label=f"{au.replace('_r','')} — {AU_NAMES[au]}")
    ax.fill_between(t_common, mn-sem, mn+sem, alpha=0.18, color=color)

ax.axvline(0, color='black', lw=2, linestyle='--', label='t=0: inicio respuesta')
ax.axvspan(-WINDOW_BEFORE, 0, alpha=0.07, color='#4C72B0', label='Pregunta')
ax.axvspan(0, WINDOW_AFTER, alpha=0.07, color='#DD8452', label='Respuesta')
ax.set_xlabel('Tiempo relativo al inicio de respuesta (s)')
ax.set_ylabel('Intensidad AU (media ± SEM)')
ax.set_title(f'Event-Locked Average de AUs alineado en t=0 (n={len(list(aligned.values())[0])} clips)')
ax.legend(ncol=2, fontsize=8)
plt.tight_layout(); savefig('fig29_event_locked_au.png'); plt.show()


In [ ]:
t1 = pd.DataFrame({'Métrica': [
        'Total fotogramas', 'Clips analizados', 'Duración media clip (s)',
        'Frames en ventana pregunta', 'Frames en ventana respuesta',
        'Confidence media global', 'Tasa de success media (%)',
        'FPS nominal', 'Sesión 1 (clips)', 'Sesión 2 (clips)'],
    'Valor': [
        f"{len(df_all):,}",
        str(df_all['id'].nunique()),
        f"{df_all.groupby('id')['timestamp'].max().mean():.2f}",
        f"{(df_all['estado']=='pregunta').sum():,}",
        f"{(df_all['estado']=='respuesta').sum():,}",
        f"{df_all['confidence'].mean():.4f}",
        f"{df_all['success'].mean()*100:.2f}",
        "30",
        str((df_all.groupby('id')['sesion'].first()==1).sum()),
        str((df_all.groupby('id')['sesion'].first()==2).sum()),
    ]})
print("Tabla 1 — Resumen del dataset:"); print(t1.to_string(index=False))
t1.to_csv(os.path.join(OUTPUT_DIR, 'tabla1_resumen_dataset.csv'), index=False)

df_stats.to_csv(os.path.join(OUTPUT_DIR, 'tabla2_tests_estadisticos_au.csv'), index=False)
print("\nTabla 2 exportada: tabla2_tests_estadisticos_au.csv")

metrics_df = pd.DataFrame({'k': list(k_range), 'Inercia': inertias,
                            'Silhouette': sil_sc, 'CH': ch_sc, 'DB': db_sc})
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'tabla3_metricas_clustering.csv'), index=False)
print("Tabla 3 exportada: tabla3_metricas_clustering.csv")

df_snr.to_csv(os.path.join(OUTPUT_DIR, 'tabla4_snr_viabilidad.csv'), index=False)
print("Tabla 4 exportada: tabla4_snr_viabilidad.csv")

cluster_profile = df_feat.groupby('cluster_km').agg(
    n_frames  = ('frame','count'),
    pct_resp  = ('estado', lambda x: (x=='respuesta').mean()*100),
    conf_media= ('confidence','mean'),
    **{a.replace('_r','')+'_mean': (a,'mean') for a in AU_COLS_R},
).round(4)
cluster_profile.to_csv(os.path.join(OUTPUT_DIR, 'tabla5_perfil_clusters.csv'))
print("Tabla 5 exportada: tabla5_perfil_clusters.csv")

print(f"\n✓ Todo exportado a {OUTPUT_DIR}")
